# Optuna Hyperparameter Optimization - CEM with Per-Concept Weights

**Runtime:** ~2 hours (50 trials)

This notebook:
1. Uses Optuna to optimize hyperparameters for CEM with per-concept class weights
2. Combines LDAM Loss + WeightedRandomSampler + Per-Concept BCE weights
3. Finds threshold achieving 75% recall on validation set
4. Maximizes F1 score as the optimization objective

**New Hyperparameters (per-concept weighting):**
- `use_per_concept_weights`: Enable/disable per-concept weighting
- `concept_weight_method`: "inverse_freq" or "sqrt_inverse"
- `concept_weight_clip_max`: Maximum weight clipping (10.0 - 100.0)

**Prerequisites:** Run `0c_prepare_max_alt_dataset.ipynb` first!

## Section 0: Setup & Configuration

In [ ]:
# Imports
import os
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    balanced_accuracy_score,
    classification_report,
    precision_recall_curve,
)

import optuna
import matplotlib.pyplot as plt

print("\u2713 All imports successful")

In [ ]:
# Optuna settings
N_TRIALS = 50
TIMEOUT_HOURS = 2.0
BASE_SEED = 42
TARGET_RECALL = 0.75  # For threshold optimization

# Set base seed for reproducibility
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
pl.seed_everything(BASE_SEED)

print(f"\u2713 Optuna settings configured")
print(f"  N_TRIALS: {N_TRIALS}")
print(f"  TIMEOUT_HOURS: {TIMEOUT_HOURS}")
print(f"  BASE_SEED: {BASE_SEED}")
print(f"  TARGET_RECALL: {TARGET_RECALL}")
print(f"  Note: Per-trial seeds will be generated deterministically from trial.number")

In [ ]:
# Detect device
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("\u2713 Using MacBook GPU (MPS)")
elif torch.cuda.is_available():
    DEVICE = "cuda"
    print("\u2713 Using CUDA GPU")
else:
    DEVICE = "cpu"
    print("\u26a0 Using CPU")

In [ ]:
# Define paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_PROCESSED = os.path.join(PROJECT_ROOT, "data/processed")
DATASET_DIR = os.path.join(DATA_PROCESSED, "max_alternative_attention_pipeline")
OUTPUT_DIR = "outputs_cem_per_concept_weights_optuna"

print("\u2713 Paths configured")
print(f"  Dataset dir: {DATASET_DIR}")
print(f"  Output dir: {OUTPUT_DIR}")

In [ ]:
# Define 21 BDI-II concept names
CONCEPT_NAMES = [
    "Sadness", "Pessimism", "Past failure", "Loss of pleasure",
    "Guilty feelings", "Punishment feelings", "Self-dislike", "Self-criticalness",
    "Suicidal thoughts or wishes", "Crying", "Agitation", "Loss of interest",
    "Indecisiveness", "Worthlessness", "Loss of energy", "Changes in sleeping pattern",
    "Irritability", "Changes in appetite", "Concentration difficulty",
    "Tiredness or fatigue", "Loss of interest in sex"
]
N_CONCEPTS = len(CONCEPT_NAMES)

print(f"\u2713 Defined {N_CONCEPTS} BDI-II concepts")

In [ ]:
# Fixed hyperparameters (not optimized)
FIXED_PARAMS = {
    "embedding_dim": 384,
    "n_concepts": 21,
    "n_tasks": 1,
    "batch_size_train": 32,
    "batch_size_eval": 64,
    "max_epochs": 100,
    "shared_prob_gen": True,
}

print("\u2713 Fixed hyperparameters configured:")
for key, value in FIXED_PARAMS.items():
    print(f"  {key}: {value}")

## Section 1: Load Preprocessed Data

In [ ]:
# Load training data
print("Loading preprocessed datasets...")

train_data = np.load(os.path.join(DATASET_DIR, "train_data.npz"))
X_train = train_data['X']
C_train = train_data['C']
y_train = train_data['y']
train_subject_ids = train_data['subject_ids']

print(f"\u2713 Loaded training data: {X_train.shape}")

In [ ]:
# Load validation data
val_data = np.load(os.path.join(DATASET_DIR, "val_data.npz"))
X_val = val_data['X']
C_val = val_data['C']
y_val = val_data['y']
val_subject_ids = val_data['subject_ids']

print(f"\u2713 Loaded validation data: {X_val.shape}")

In [ ]:
# Load class weights
with open(os.path.join(DATASET_DIR, "class_weights.json"), 'r') as f:
    class_info = json.load(f)

n_positive = class_info['n_positive']
n_negative = class_info['n_negative']
pos_weight = class_info['pos_weight']

print(f"\u2713 Loaded class weights:")
print(f"  Negative: {n_negative}, Positive: {n_positive}")
print(f"  Ratio: 1:{pos_weight:.2f}")

print("\n\u26a0 Test data will be loaded ONLY after optimization completes!")

In [ ]:
# Display per-concept class distribution
print("\n" + "="*70)
print("      PER-CONCEPT CLASS DISTRIBUTION (Training Set)")
print("="*70)

for i, name in enumerate(CONCEPT_NAMES):
    n_pos = int(C_train[:, i].sum())
    n_neg = int(C_train.shape[0] - n_pos)
    ratio = n_neg / n_pos if n_pos > 0 else float('inf')
    print(f"{name:<30}: {n_pos:>3} pos / {n_neg:>3} neg (ratio: {ratio:.2f})")

## Section 2: Helper Functions

In [ ]:
def compute_per_concept_pos_weights(C_train, method="inverse_freq", clip_max=50.0):
    """
    Compute pos_weight for each concept based on class distribution.

    Args:
        C_train: numpy array of shape (n_samples, n_concepts)
        method: "inverse_freq" | "sqrt_inverse"
        clip_max: maximum weight to prevent instability

    Returns:
        torch.Tensor of shape [n_concepts]
    """
    n_samples, n_concepts = C_train.shape
    pos_weights = []

    for i in range(n_concepts):
        n_positive = C_train[:, i].sum()
        n_negative = n_samples - n_positive

        if n_positive == 0:
            weight = 1.0
        elif method == "inverse_freq":
            weight = n_negative / n_positive
        elif method == "sqrt_inverse":
            weight = np.sqrt(n_negative / n_positive)
        else:
            weight = n_negative / n_positive

        pos_weights.append(min(weight, clip_max))

    return torch.tensor(pos_weights, dtype=torch.float32)

print("\u2713 Per-concept weight computation function defined")

In [ ]:
def find_threshold_for_target_recall(y_true, y_prob, target_recall=0.75):
    """
    Find threshold that achieves target recall with best precision.

    For depression screening: Prioritize catching cases (recall) over precision.

    Args:
        y_true: True labels
        y_prob: Predicted probabilities
        target_recall: Minimum recall required (default: 0.75)

    Returns:
        best_threshold: Threshold achieving target recall
        achieved_recall: Actual recall achieved
        precision: Precision at that threshold
    """
    best_precision = 0
    best_threshold = 0.5
    achieved_recall = 0

    for threshold in np.arange(0.01, 0.99, 0.01):  # Fine-grained search
        y_pred = (y_prob >= threshold).astype(int)

        # Skip if no positives predicted
        if np.sum(y_pred) == 0:
            continue

        try:
            recall = recall_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred)
        except:
            continue

        # Only consider thresholds that meet recall target
        if recall >= target_recall:
            if precision > best_precision:
                best_precision = precision
                best_threshold = threshold
                achieved_recall = recall

    # If no threshold achieves target recall, return best recall achieved
    if achieved_recall == 0:
        best_recall = 0
        for threshold in np.arange(0.01, 0.99, 0.01):
            y_pred = (y_prob >= threshold).astype(int)
            if np.sum(y_pred) == 0:
                continue
            try:
                recall = recall_score(y_true, y_pred)
                precision = precision_score(y_true, y_pred)
            except:
                continue
            if recall > best_recall:
                best_recall = recall
                best_precision = precision
                best_threshold = threshold
                achieved_recall = recall

    return best_threshold, achieved_recall, best_precision

print("\u2713 Threshold optimization function defined")

## Section 3: Model Definition

In [ ]:
# PyTorch Dataset
class CEMDataset(Dataset):
    def __init__(self, X, C, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.C = torch.tensor(C, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.C[idx]

# Create datasets (DataLoaders will be created per trial)
train_dataset = CEMDataset(X_train, C_train, y_train)
val_dataset = CEMDataset(X_val, C_val, y_val)

# Validation loader (fixed, no sampling)
val_loader = DataLoader(val_dataset, batch_size=FIXED_PARAMS['batch_size_eval'], shuffle=False)

print("\u2713 Datasets created")
print("  Train DataLoader will be created per trial (with/without sampler)")
print("  Validation DataLoader created (fixed)")

In [ ]:
# LDAM Loss (for class imbalance)
class LDAMLoss(nn.Module):
    """
    Label-Distribution-Aware Margin (LDAM) Loss for long-tailed recognition.

    Creates class-dependent margins to make decision boundaries harder for minority classes.
    """
    def __init__(self, n_positive, n_negative, max_margin=0.5, scale=30):
        super(LDAMLoss, self).__init__()
        self.max_margin = max_margin
        self.scale = scale

        # Compute class frequencies
        total = n_positive + n_negative
        freq_pos = n_positive / total
        freq_neg = n_negative / total

        # Compute margins: minority class gets larger margin
        margin_pos = max_margin * (freq_pos ** (-0.25))
        margin_neg = max_margin * (freq_neg ** (-0.25))

        self.register_buffer('margin_pos', torch.tensor(margin_pos))
        self.register_buffer('margin_neg', torch.tensor(margin_neg))

    def forward(self, logits, targets):
        logits = logits.view(-1)
        targets = targets.view(-1).float()

        # Apply class-dependent margins
        margin = targets * self.margin_pos + (1 - targets) * (-self.margin_neg)
        adjusted_logits = (logits - margin) * self.scale

        return F.binary_cross_entropy_with_logits(adjusted_logits, targets, reduction='mean')

print("\u2713 LDAMLoss defined")

In [ ]:
# Custom CEM Implementation with per-concept weights support
class CustomCEM(pl.LightningModule):
    """
    Custom Concept Embedding Model (CEM) implementation.

    Architecture:
      X -> concept_extractor -> context_layers -> prob_generator -> dual_embeddings -> task_classifier -> y

    Supports per-concept pos_weight for BCEWithLogitsLoss to handle concept-level imbalance.
    """
    def __init__(
        self,
        n_concepts=21,
        emb_size=128,
        input_dim=384,
        shared_prob_gen=True,
        intervention_prob=0.25,
        concept_loss_weight=1.0,
        learning_rate=0.01,
        weight_decay=4e-05,
        use_ldam_loss=True,
        n_positive=83,
        n_negative=403,
        ldam_max_margin=0.5,
        ldam_scale=30,
        concept_pos_weights=None,  # Per-concept positive class weights
    ):
        super().__init__()
        self.save_hyperparameters()

        self.n_concepts = n_concepts
        self.emb_size = emb_size
        self.intervention_prob = intervention_prob
        self.concept_loss_weight = concept_loss_weight

        # Stage 1: Concept Extractor (X -> Pre-Concept Features)
        self.concept_extractor = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 256)  # Pre-concept features
        )

        # Stage 2: Context Generators (Features -> Dual Embeddings)
        self.context_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(256, emb_size * 2),  # Dual embeddings (true/false)
                nn.LeakyReLU()
            ) for _ in range(n_concepts)
        ])

        # Stage 3: Probability Generator (Contexts -> Concept Probabilities)
        if shared_prob_gen:
            self.prob_generator = nn.Linear(emb_size * 2, 1)
        else:
            self.prob_generators = nn.ModuleList([
                nn.Linear(emb_size * 2, 1) for _ in range(n_concepts)
            ])

        self.shared_prob_gen = shared_prob_gen

        # Stage 4: Task Classifier (Concept Embeddings -> Task Output)
        self.task_classifier = nn.Sequential(
            nn.Linear(n_concepts * emb_size, 128),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)  # Binary classification
        )

        # Concept loss (support per-concept weights)
        if concept_pos_weights is not None:
            self.register_buffer('concept_pos_weights', concept_pos_weights)
            self.concept_loss_fn = nn.BCEWithLogitsLoss(pos_weight=concept_pos_weights)
        else:
            self.concept_loss_fn = nn.BCEWithLogitsLoss()

        # Task loss
        if use_ldam_loss:
            self.task_loss_fn = LDAMLoss(n_positive, n_negative, ldam_max_margin, ldam_scale)
        else:
            self.task_loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, x, c_true=None, train=False):
        # Step 1: Extract pre-concept features
        pre_features = self.concept_extractor(x)  # (B, 256)

        # Step 2: Generate contexts and probabilities per concept
        contexts = []
        c_logits_list = []

        for i, context_layer in enumerate(self.context_layers):
            context = context_layer(pre_features)  # (B, emb_size*2)

            # Get probability logit
            if self.shared_prob_gen:
                logit = self.prob_generator(context)  # (B, 1)
            else:
                logit = self.prob_generators[i](context)

            contexts.append(context)
            c_logits_list.append(logit)

        c_logits = torch.cat(c_logits_list, dim=1)  # (B, 21)
        c_probs = torch.sigmoid(c_logits)           # (B, 21)

        # Step 3: Apply intervention (optional during training)
        if train and self.intervention_prob > 0 and c_true is not None:
            intervention_mask = torch.bernoulli(
                torch.ones_like(c_probs) * self.intervention_prob
            )
            c_probs = c_probs * (1 - intervention_mask) + c_true * intervention_mask

        # Step 4: Mix dual embeddings based on probabilities
        concept_embeddings = []
        for i, context in enumerate(contexts):
            emb_true = context[:, :self.emb_size]       # First half
            emb_false = context[:, self.emb_size:]      # Second half

            prob = c_probs[:, i:i+1]  # (B, 1)
            mixed_emb = emb_true * prob + emb_false * (1 - prob)
            concept_embeddings.append(mixed_emb)

        c_embeddings = torch.cat(concept_embeddings, dim=1)  # (B, 21*emb_size)

        # Step 5: Task prediction
        y_logits = self.task_classifier(c_embeddings)  # (B, 1)

        return c_logits, y_logits

    def training_step(self, batch, batch_idx):
        x, y, c_true = batch
        c_logits, y_logits = self.forward(x, c_true=c_true, train=True)

        # Task loss (LDAM or BCE)
        task_loss = self.task_loss_fn(y_logits.squeeze(), y.squeeze())

        # Concept loss (BCE with per-concept weights if enabled)
        concept_loss = self.concept_loss_fn(c_logits, c_true)

        # Combined loss
        loss = task_loss + self.concept_loss_weight * concept_loss

        # Logging
        self.log('train_loss', loss, on_epoch=True)
        self.log('train_task_loss', task_loss, on_epoch=True)
        self.log('train_concept_loss', concept_loss, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y, c_true = batch
        c_logits, y_logits = self.forward(x, c_true=c_true, train=False)

        # Task loss
        task_loss = self.task_loss_fn(y_logits.squeeze(), y.squeeze())

        # Concept loss
        concept_loss = self.concept_loss_fn(c_logits, c_true)

        # Combined loss
        loss = task_loss + self.concept_loss_weight * concept_loss

        # Logging
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_task_loss', task_loss, on_epoch=True)
        self.log('val_concept_loss', concept_loss, on_epoch=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay
        )

print("\u2713 Custom CEM model defined (with per-concept weight support)")

## Section 4: Objective Function

In [ ]:
def objective(trial):
    """
    Optuna objective function to maximize validation F1 while achieving target recall.

    Strategy:
      1. Train model with sampled hyperparameters
      2. Find threshold that achieves TARGET_RECALL on validation set
      3. Return F1 score at that threshold

    Returns:
        float: F1 score on validation set
    """
    # ============================================================================
    # STEP 0: Per-Trial Deterministic Seeding
    # ============================================================================
    trial_seed = BASE_SEED + trial.number

    np.random.seed(trial_seed)
    torch.manual_seed(trial_seed)
    pl.seed_everything(trial_seed, workers=True)

    trial.set_user_attr('trial_seed', trial_seed)

    # ============================================================================
    # STEP 1: Sample hyperparameters (existing + new per-concept params)
    # ============================================================================
    # Existing hyperparameters
    use_ldam = trial.suggest_categorical('use_ldam_loss', [True, False])
    ldam_margin = trial.suggest_float('ldam_max_margin', 0.1, 1.0)
    ldam_scale = trial.suggest_int('ldam_scale', 10, 50)
    use_sampler = trial.suggest_categorical('use_weighted_sampler', [True, False])
    lr = trial.suggest_float('learning_rate', 0.001, 0.05, log=True)
    concept_weight = trial.suggest_float('concept_loss_weight', 0.5, 2.0)
    emb_size = trial.suggest_categorical('emb_size', [64, 128, 256])
    intervention = trial.suggest_float('intervention_prob', 0.0, 0.5)
    wd = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    # NEW: Per-concept weighting hyperparameters
    use_per_concept_weights = trial.suggest_categorical('use_per_concept_weights', [True, False])
    concept_weight_method = trial.suggest_categorical('concept_weight_method', ['inverse_freq', 'sqrt_inverse'])
    concept_weight_clip_max = trial.suggest_float('concept_weight_clip_max', 10.0, 100.0)

    # Log all hyperparameters to trial attributes
    trial.set_user_attr('use_ldam_loss', use_ldam)
    trial.set_user_attr('ldam_max_margin', ldam_margin)
    trial.set_user_attr('ldam_scale', ldam_scale)
    trial.set_user_attr('use_weighted_sampler', use_sampler)
    trial.set_user_attr('learning_rate', lr)
    trial.set_user_attr('concept_loss_weight', concept_weight)
    trial.set_user_attr('emb_size', emb_size)
    trial.set_user_attr('intervention_prob', intervention)
    trial.set_user_attr('weight_decay', wd)
    trial.set_user_attr('use_per_concept_weights', use_per_concept_weights)
    trial.set_user_attr('concept_weight_method', concept_weight_method)
    trial.set_user_attr('concept_weight_clip_max', concept_weight_clip_max)

    # ============================================================================
    # STEP 2: Compute per-concept weights (if enabled)
    # ============================================================================
    if use_per_concept_weights:
        concept_pos_weights = compute_per_concept_pos_weights(
            C_train,
            method=concept_weight_method,
            clip_max=concept_weight_clip_max
        )
    else:
        concept_pos_weights = None

    # ============================================================================
    # STEP 3: Create DataLoader with or without sampler
    # ============================================================================
    if use_sampler:
        class_sample_counts = np.bincount(y_train.astype(int))
        weights = 1.0 / class_sample_counts
        sample_weights = weights[y_train.astype(int)]

        train_sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
            generator=torch.Generator().manual_seed(trial_seed)
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=FIXED_PARAMS['batch_size_train'],
            sampler=train_sampler,
            worker_init_fn=lambda worker_id: np.random.seed(trial_seed + worker_id)
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=FIXED_PARAMS['batch_size_train'],
            shuffle=True,
            generator=torch.Generator().manual_seed(trial_seed),
            worker_init_fn=lambda worker_id: np.random.seed(trial_seed + worker_id)
        )

    # ============================================================================
    # STEP 4: Create model
    # ============================================================================
    model = CustomCEM(
        n_concepts=FIXED_PARAMS['n_concepts'],
        emb_size=emb_size,
        input_dim=FIXED_PARAMS['embedding_dim'],
        shared_prob_gen=FIXED_PARAMS['shared_prob_gen'],
        intervention_prob=intervention,
        concept_loss_weight=concept_weight,
        learning_rate=lr,
        weight_decay=wd,
        use_ldam_loss=use_ldam,
        n_positive=n_positive,
        n_negative=n_negative,
        ldam_max_margin=ldam_margin,
        ldam_scale=ldam_scale,
        concept_pos_weights=concept_pos_weights,
    )

    # ============================================================================
    # STEP 5: Setup trainer with EarlyStopping
    # ============================================================================
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=15,
        mode='min',
        verbose=False
    )

    trainer = pl.Trainer(
        max_epochs=FIXED_PARAMS['max_epochs'],
        accelerator=DEVICE,
        devices=1,
        callbacks=[early_stop_callback],
        enable_progress_bar=False,
        logger=False,
        enable_checkpointing=False,
    )

    # ============================================================================
    # STEP 6: Train
    # ============================================================================
    try:
        trainer.fit(model, train_loader, val_loader)
    except optuna.TrialPruned:
        raise

    # ============================================================================
    # STEP 7: Run validation inference
    # ============================================================================
    model.eval()
    device_obj = torch.device(DEVICE)
    model = model.to(device_obj)

    y_prob_val = []
    y_true_val = []

    with torch.no_grad():
        for x_batch, y_batch, c_batch in val_loader:
            x_batch = x_batch.to(device_obj)
            c_batch = c_batch.to(device_obj)

            c_logits, y_logits = model(x_batch)

            y_probs = torch.sigmoid(y_logits).cpu().squeeze().numpy()
            y_true_batch = y_batch.cpu().numpy().astype(int)

            if y_probs.ndim == 0:
                y_prob_val.append(float(y_probs))
                y_true_val.append(int(y_true_batch))
            else:
                y_prob_val.extend(y_probs.tolist())
                y_true_val.extend(y_true_batch.tolist())

    y_prob_val = np.array(y_prob_val)
    y_true_val = np.array(y_true_val)

    # ============================================================================
    # STEP 8: Find threshold achieving TARGET_RECALL on validation set
    # ============================================================================
    best_threshold, achieved_recall, precision = find_threshold_for_target_recall(
        y_true_val, y_prob_val, target_recall=TARGET_RECALL
    )

    # Calculate F1 for this threshold
    if achieved_recall > 0 and precision > 0:
        f1 = 2 * (precision * achieved_recall) / (precision + achieved_recall)
    else:
        f1 = 0.0

    # Calculate MCC as well
    y_pred_val = (y_prob_val >= best_threshold).astype(int)
    mcc = matthews_corrcoef(y_true_val, y_pred_val)

    # ============================================================================
    # STEP 9: Log all metrics to trial user attributes
    # ============================================================================
    trial.set_user_attr('best_threshold', float(best_threshold))
    trial.set_user_attr('achieved_recall', float(achieved_recall))
    trial.set_user_attr('precision', float(precision))
    trial.set_user_attr('f1_score', float(f1))
    trial.set_user_attr('mcc', float(mcc))
    trial.set_user_attr('target_recall', float(TARGET_RECALL))

    # Log training info
    if hasattr(trainer, 'callback_metrics'):
        metrics = trainer.callback_metrics
        if 'val_loss' in metrics:
            trial.set_user_attr('final_val_loss', float(metrics['val_loss']))
        if 'train_loss' in metrics:
            trial.set_user_attr('final_train_loss', float(metrics['train_loss']))

    trial.set_user_attr('num_epochs_trained', trainer.current_epoch)
    trial.set_user_attr('early_stopped', trainer.current_epoch < FIXED_PARAMS['max_epochs'])

    # ============================================================================
    # STEP 10: Return F1 score (objective to maximize)
    # ============================================================================
    return f1


print("\u2713 Objective function defined")

## Section 5: Run Optuna Study

In [ ]:
# Create Optuna study
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=BASE_SEED),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,       # Don't prune first 5 trials
        n_warmup_steps=10,        # Wait 10 epochs before pruning
        interval_steps=5,         # Check every 5 epochs
    )
)

print("="*70)
print("                 OPTUNA STUDY CREATED")
print("="*70)
print("\nConfiguration:")
print(f"  Objective:     Achieve {TARGET_RECALL:.0%} recall with maximum F1")
print("  Sampler:       TPE (Tree-structured Parzen Estimator)")
print("  Pruner:        MedianPruner (early stopping)")
print("  Search Space:  12 hyperparameters")
print("\nExisting hyperparameters:")
print("  - use_ldam_loss: [True, False]")
print("  - ldam_max_margin: [0.1, 1.0]")
print("  - ldam_scale: [10, 50]")
print("  - use_weighted_sampler: [True, False]")
print("  - learning_rate: [0.001, 0.05] (log scale)")
print("  - concept_loss_weight: [0.5, 2.0]")
print("  - emb_size: [64, 128, 256]")
print("  - intervention_prob: [0.0, 0.5]")
print("  - weight_decay: [1e-5, 1e-3] (log scale)")
print("\nNEW per-concept hyperparameters:")
print("  - use_per_concept_weights: [True, False]")
print("  - concept_weight_method: ['inverse_freq', 'sqrt_inverse']")
print("  - concept_weight_clip_max: [10.0, 100.0]")
print("="*70)

In [ ]:
# Run optimization
timeout = TIMEOUT_HOURS * 3600  # Convert to seconds

print("\n" + "="*70)
print("                STARTING OPTIMIZATION")
print("="*70)
print(f"\nSettings:")
print(f"  Max trials:        {N_TRIALS}")
print(f"  Timeout:           {TIMEOUT_HOURS} hours")
print(f"  Target recall:     {TARGET_RECALL:.0%}")
print(f"\n\u23f0 This may take several hours. Monitor progress below...\n")
print("="*70)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    timeout=timeout,
    show_progress_bar=True
)

print("\n" + "="*70)
print("                OPTIMIZATION COMPLETE")
print("="*70)
print(f"\nResults:")
print(f"  Completed trials:  {len(study.trials)}")
print(f"  Pruned trials:     {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"  Best F1:           {study.best_value:.4f}")
print("="*70)

## Section 6: Results Visualization

In [ ]:
# Display best hyperparameters
print("="*70)
print("                 BEST HYPERPARAMETERS")
print("="*70)

best_params = study.best_params
best_threshold = study.best_trial.user_attrs['best_threshold']

print("\nOptimal hyperparameters:")
for key, value in sorted(best_params.items()):
    if isinstance(value, float):
        print(f"  {key:<30} {value:.6f}")
    else:
        print(f"  {key:<30} {value}")

print(f"\n  {'best_threshold':<30} {best_threshold:.2f} (optimized on validation)")
print(f"\nValidation Performance:")
print(f"  F1 Score:                     {study.best_value:.4f}")
print(f"  MCC:                          {study.best_trial.user_attrs.get('mcc', 'N/A')}")
print(f"  Achieved Recall:              {study.best_trial.user_attrs.get('achieved_recall', 'N/A')}")
print(f"  Precision:                    {study.best_trial.user_attrs.get('precision', 'N/A')}")
print("="*70)

In [ ]:
# Save best hyperparameters
best_config = {
    **study.best_params,
    'best_threshold': float(best_threshold),
    'validation_f1': float(study.best_value),
    'validation_mcc': float(study.best_trial.user_attrs.get('mcc', 0)),
    'achieved_recall': float(study.best_trial.user_attrs.get('achieved_recall', 0)),
    'precision': float(study.best_trial.user_attrs.get('precision', 0)),
    'target_recall': float(TARGET_RECALL),
    'n_trials': len(study.trials),
    'n_pruned': len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, 'best_hyperparameters.json'), 'w') as f:
    json.dump(best_config, f, indent=4)

print(f"\u2713 Saved best hyperparameters to {OUTPUT_DIR}/best_hyperparameters.json")

In [ ]:
# Visualization: Optimization History
fig = optuna.visualization.plot_optimization_history(study)
fig.write_html(os.path.join(OUTPUT_DIR, 'optimization_history.html'))
fig.show()

print(f"\u2713 Saved optimization history plot to {OUTPUT_DIR}/optimization_history.html")

In [ ]:
# Visualization: Parameter Importances
try:
    fig = optuna.visualization.plot_param_importances(study)
    fig.write_html(os.path.join(OUTPUT_DIR, 'param_importances.html'))
    fig.show()
    print(f"\u2713 Saved parameter importance plot to {OUTPUT_DIR}/param_importances.html")
except Exception as e:
    print(f"\u26a0 Could not generate parameter importance plot: {e}")

In [ ]:
# Visualization: Parallel Coordinate Plot
try:
    fig = optuna.visualization.plot_parallel_coordinate(study)
    fig.write_html(os.path.join(OUTPUT_DIR, 'parallel_coordinate.html'))
    fig.show()
    print(f"\u2713 Saved parallel coordinate plot to {OUTPUT_DIR}/parallel_coordinate.html")
except Exception as e:
    print(f"\u26a0 Could not generate parallel coordinate plot: {e}")

## Section 7: Train Final Model with Best Params

In [ ]:
print("\n" + "="*70)
print("           TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("="*70)

# Load test data
print("\nLoading test data...")
test_data = np.load(os.path.join(DATASET_DIR, "test_data.npz"))
X_test = test_data['X']
C_test = test_data['C']
y_test = test_data['y']
test_subject_ids = test_data['subject_ids']

print(f"\u2713 Loaded test data: {X_test.shape}")

In [ ]:
# Combine train + val for final training
print("\nCombining train + validation sets...")

X_train_full = np.concatenate([X_train, X_val], axis=0)
C_train_full = np.concatenate([C_train, C_val], axis=0)
y_train_full = np.concatenate([y_train, y_val], axis=0)

print(f"\u2713 Combined: {X_train_full.shape}")

train_full_dataset = CEMDataset(X_train_full, C_train_full, y_train_full)
test_dataset = CEMDataset(X_test, C_test, y_test)

In [ ]:
# Load best hyperparameters
print("\nLoading best hyperparameters...")

with open(os.path.join(OUTPUT_DIR, 'best_hyperparameters.json'), 'r') as f:
    best_config = json.load(f)

print("\u2713 Best hyperparameters loaded:")
for key, value in sorted(best_config.items()):
    if key not in ['n_trials', 'n_pruned', 'validation_f1', 'validation_mcc', 'achieved_recall', 'precision', 'target_recall']:
        print(f"  {key:<30} {value}")

In [ ]:
# Create DataLoader with best configuration
print("\nCreating final DataLoader...")

use_sampler = best_config['use_weighted_sampler']
final_seed = BASE_SEED + 9999

if use_sampler:
    class_sample_counts = np.bincount(y_train_full.astype(int))
    weights = 1.0 / class_sample_counts
    sample_weights = weights[y_train_full.astype(int)]

    train_full_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=torch.Generator().manual_seed(final_seed)
    )

    train_full_loader = DataLoader(
        train_full_dataset,
        batch_size=FIXED_PARAMS['batch_size_train'],
        sampler=train_full_sampler,
        worker_init_fn=lambda worker_id: np.random.seed(final_seed + worker_id)
    )
    print("\u2713 Using WeightedRandomSampler")
else:
    train_full_loader = DataLoader(
        train_full_dataset,
        batch_size=FIXED_PARAMS['batch_size_train'],
        shuffle=True,
        generator=torch.Generator().manual_seed(final_seed),
        worker_init_fn=lambda worker_id: np.random.seed(final_seed + worker_id)
    )
    print("\u2713 Using standard shuffle")

test_loader = DataLoader(test_dataset, batch_size=FIXED_PARAMS['batch_size_eval'], shuffle=False)

In [ ]:
# Compute per-concept weights for final model (if enabled)
if best_config['use_per_concept_weights']:
    final_concept_pos_weights = compute_per_concept_pos_weights(
        C_train_full,
        method=best_config['concept_weight_method'],
        clip_max=best_config['concept_weight_clip_max']
    )
    print(f"\u2713 Per-concept weights computed (method={best_config['concept_weight_method']}, clip={best_config['concept_weight_clip_max']})")
else:
    final_concept_pos_weights = None
    print("\u2713 Per-concept weights disabled")

In [ ]:
# Seed for final model
np.random.seed(final_seed)
torch.manual_seed(final_seed)
pl.seed_everything(final_seed, workers=True)

# Create final model
print("\nInitializing final model...")

final_model = CustomCEM(
    n_concepts=FIXED_PARAMS['n_concepts'],
    emb_size=best_config['emb_size'],
    input_dim=FIXED_PARAMS['embedding_dim'],
    shared_prob_gen=FIXED_PARAMS['shared_prob_gen'],
    intervention_prob=best_config['intervention_prob'],
    concept_loss_weight=best_config['concept_loss_weight'],
    learning_rate=best_config['learning_rate'],
    weight_decay=best_config['weight_decay'],
    use_ldam_loss=best_config['use_ldam_loss'],
    n_positive=n_positive,
    n_negative=n_negative,
    ldam_max_margin=best_config['ldam_max_margin'],
    ldam_scale=best_config['ldam_scale'],
    concept_pos_weights=final_concept_pos_weights,
)

print("\u2713 Final model initialized")

In [ ]:
# Setup trainer
print("\nSetting up trainer...")

final_checkpoint = ModelCheckpoint(
    monitor="train_loss",
    dirpath=os.path.join(OUTPUT_DIR, "final_model"),
    filename="final-cem-per-concept-{epoch:02d}-{train_loss:.2f}",
    save_top_k=1,
    mode="min"
)

final_trainer = pl.Trainer(
    max_epochs=FIXED_PARAMS['max_epochs'],
    accelerator=DEVICE,
    devices=1,
    callbacks=[final_checkpoint],
    enable_progress_bar=True,
    logger=CSVLogger(save_dir=os.path.join(OUTPUT_DIR, "logs"), name="final_model"),
)

print("\nStarting final model training...\n")
final_trainer.fit(final_model, train_full_loader)
print("\n\u2713 Training complete!")

## Section 8: Test Evaluation

In [ ]:
# Test set inference
print("\n" + "="*70)
print("                  FINAL MODEL - TEST SET EVALUATION")
print("="*70)

print("\nRunning inference...")

final_model.eval()
device_obj = torch.device(DEVICE)
final_model = final_model.to(device_obj)

y_true_test = []
y_prob_test = []
concept_probs_test = []

with torch.no_grad():
    for x_batch, y_batch, c_batch in test_loader:
        x_batch = x_batch.to(device_obj)
        c_logits, y_logits = final_model(x_batch)

        c_probs = torch.sigmoid(c_logits).cpu().numpy()
        y_probs = torch.sigmoid(y_logits).cpu().squeeze().numpy()
        y_true_batch = y_batch.cpu().numpy().astype(int)

        if y_probs.ndim == 0:
            y_prob_test.append(float(y_probs))
            y_true_test.append(int(y_true_batch))
            concept_probs_test.append(c_probs.squeeze())
        else:
            y_prob_test.extend(y_probs.tolist())
            y_true_test.extend(y_true_batch.tolist())
            concept_probs_test.extend(c_probs.tolist())

y_true_test = np.array(y_true_test)
y_prob_test = np.array(y_prob_test)
concept_probs_test = np.array(concept_probs_test)

print(f"\u2713 Predictions for {len(y_true_test)} samples")

In [ ]:
# Apply best threshold and compute metrics
best_threshold = best_config['best_threshold']
print(f"\nApplying threshold: {best_threshold:.2f}")

y_pred_test = (y_prob_test >= best_threshold).astype(int)

cm = confusion_matrix(y_true_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()

test_accuracy = accuracy_score(y_true_test, y_pred_test)
test_balanced_acc = balanced_accuracy_score(y_true_test, y_pred_test)
test_roc_auc = roc_auc_score(y_true_test, y_prob_test)
test_mcc = matthews_corrcoef(y_true_test, y_pred_test)
test_f1 = f1_score(y_true_test, y_pred_test)
test_precision = precision_score(y_true_test, y_pred_test) if (tp + fp) > 0 else 0.0
test_recall = recall_score(y_true_test, y_pred_test) if (tp + fn) > 0 else 0.0

# Display results
print("\n" + "="*70)
print("                    TEST SET RESULTS")
print("="*70)
print(f"\n{'CONFUSION MATRIX':^50}")
print("="*50)
print(f"{'':>20} | {'Predicted Neg':^15} | {'Predicted Pos':^15}")
print("-"*50)
print(f"{'Actual Negative':>20} | {f'TN = {tn}':^15} | {f'FP = {fp}':^15}")
print(f"{'Actual Positive':>20} | {f'FN = {fn}':^15} | {f'TP = {tp}':^15}")
print("="*50)

n_pos = int(np.sum(y_true_test))
n_neg = int(len(y_true_test) - n_pos)

print(f"\n  TP: {tp}/{n_pos} ({100*tp/n_pos if n_pos > 0 else 0:.1f}% caught)")
print(f"  FN: {fn}/{n_pos} ({100*fn/n_pos if n_pos > 0 else 0:.1f}% missed)")

print(f"\nMetrics:")
print(f"  MCC:              {test_mcc:.4f}")
print(f"  F1:               {test_f1:.4f}")
print(f"  Recall:           {test_recall:.4f}")
print(f"  Precision:        {test_precision:.4f}")
print(f"  ROC-AUC:          {test_roc_auc:.4f}")
print(f"  Accuracy:         {test_accuracy:.4f}")
print(f"  Balanced Acc:     {test_balanced_acc:.4f}")

print("\n" + classification_report(y_true_test, y_pred_test, target_names=['Negative', 'Positive']))
print("="*70)

In [ ]:
# Visualize probability distributions
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(y_prob_test[y_true_test == 0], bins=20, alpha=0.5, label='Negative')
plt.hist(y_prob_test[y_true_test == 1], bins=20, alpha=0.5, label='Positive')
plt.axvline(x=best_threshold, color='red', linestyle='--', label=f'Threshold = {best_threshold:.2f}')
plt.legend()
plt.title("Predicted Probabilities Distribution (Test Set)")
plt.xlabel("Predicted Probability")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
precision_curve, recall_curve, thresholds = precision_recall_curve(y_true_test, y_prob_test)
plt.plot(thresholds, precision_curve[:-1], label='Precision')
plt.plot(thresholds, recall_curve[:-1], label='Recall')
plt.axvline(x=best_threshold, color='red', linestyle='--', label=f'Threshold = {best_threshold:.2f}')
plt.legend()
plt.title("Precision and Recall vs. Threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'test_probability_distribution.png'), dpi=150)
plt.show()

print(f"\u2713 Saved probability distribution plot to {OUTPUT_DIR}/test_probability_distribution.png")

## Section 9: Save Results

In [ ]:
# Save final results
print("\nSaving results...")

final_results = {
    'optimization_summary': {
        'n_trials': best_config['n_trials'],
        'best_validation_f1': best_config['validation_f1'],
        'best_validation_mcc': best_config['validation_mcc'],
        'target_recall': best_config['target_recall'],
    },
    'best_hyperparameters': {k: v for k, v in best_config.items()
                             if k not in ['n_trials', 'n_pruned', 'validation_f1', 'validation_mcc', 'achieved_recall', 'precision', 'target_recall']},
    'test_metrics': {
        'threshold': float(best_threshold),
        'mcc': float(test_mcc),
        'f1': float(test_f1),
        'recall': float(test_recall),
        'precision': float(test_precision),
        'roc_auc': float(test_roc_auc),
        'accuracy': float(test_accuracy),
        'balanced_accuracy': float(test_balanced_acc),
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    }
}

with open(os.path.join(OUTPUT_DIR, 'final_test_results.json'), 'w') as f:
    json.dump(final_results, f, indent=4)

# Save predictions
predictions_df = pd.DataFrame({
    'subject_id': test_subject_ids,
    'y_true': y_true_test,
    'y_pred': y_pred_test,
    'y_prob': y_prob_test
})

for i, concept_name in enumerate(CONCEPT_NAMES):
    predictions_df[concept_name] = concept_probs_test[:, i]

predictions_df.to_csv(os.path.join(OUTPUT_DIR, 'final_test_predictions.csv'), index=False)

print(f"\u2713 Saved to {OUTPUT_DIR}/")

In [ ]:
print("\n" + "="*70)
print("              OPTIMIZATION COMPLETE")
print("="*70)

print("\n\U0001F4CA SUMMARY:")
print(f"  Trials:              {best_config['n_trials']}")
print(f"  Best val F1:         {best_config['validation_f1']:.4f}")
print(f"  Best val MCC:        {best_config['validation_mcc']:.4f}")

print("\n\U0001F3C6 BEST HYPERPARAMETERS:")
print(f"  Embedding size:      {best_config['emb_size']}")
print(f"  Learning rate:       {best_config['learning_rate']:.6f}")
print(f"  Use LDAM:            {best_config['use_ldam_loss']}")
print(f"  Use sampler:         {best_config['use_weighted_sampler']}")
print(f"  Per-concept weights: {best_config['use_per_concept_weights']}")
if best_config['use_per_concept_weights']:
    print(f"    Method:            {best_config['concept_weight_method']}")
    print(f"    Clip max:          {best_config['concept_weight_clip_max']:.1f}")

print("\n\U0001F3AF TEST PERFORMANCE:")
print(f"  MCC:                 {test_mcc:.4f}")
print(f"  F1:                  {test_f1:.4f}")
print(f"  Recall:              {test_recall:.4f} ({tp}/{n_pos} caught)")
print(f"  Precision:           {test_precision:.4f}")

print("\n\U0001F4C1 FILES:")
print(f"  Best params:         {OUTPUT_DIR}/best_hyperparameters.json")
print(f"  Test results:        {OUTPUT_DIR}/final_test_results.json")
print(f"  Predictions:         {OUTPUT_DIR}/final_test_predictions.csv")
print(f"  Model:               {OUTPUT_DIR}/final_model/")
print(f"  Optimization plot:   {OUTPUT_DIR}/optimization_history.html")
print(f"  Param importance:    {OUTPUT_DIR}/param_importances.html")

print("\n\u2705 Final model ready for deployment!")
print("="*70)